# Project 84 — Local Slide Deck Explainer
## Slide Text → Structured Notes → Speaker Script

**Stack:** LangChain · Ollama · Pydantic · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama pydantic pandas

## Step 1 — Simulated Slide Deck

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field
import json

llm = ChatOllama(model="qwen3:8b", temperature=0.3)

slides = [
    {"slide": 1, "title": "Q1 2025 Results",
     "content": "Revenue: $4.2M (+18% YoY)\nARR: $15.8M\nNet retention: 112%\nNew logos: 23"},
    {"slide": 2, "title": "Product Highlights",
     "content": "Launched AI Assistant (Jan)\nMobile app v3.0 (Feb)\nAPI v2 with webhooks (Mar)\n"
                "NPS improved from 42 to 58"},
    {"slide": 3, "title": "Customer Wins",
     "content": "Enterprise: TechCorp ($500K ACV), DataFlow ($350K ACV)\n"
                "Mid-Market: 12 new accounts ($1.2M pipeline)\nChurn: 3 accounts ($180K lost)"},
    {"slide": 4, "title": "Engineering Metrics",
     "content": "Deployment frequency: 4x/week\nMTTR: 45min (prev 2hrs)\n"
                "Test coverage: 82%\nTech debt ratio: 15% of sprint capacity"},
    {"slide": 5, "title": "Q2 Roadmap",
     "content": "AI-powered analytics dashboard\nSOC2 Type II certification\n"
                "European data center\nSelf-serve onboarding\nTarget: $5M revenue"},
]
print(f"Slide deck: {len(slides)} slides")
for s in slides:
    print(f"  Slide {s['slide']}: {s['title']}")

## Step 2 — Generate Structured Notes

In [ ]:
class SlideAnalysis(BaseModel):
    slide_number: int
    title: str
    key_points: list[str]
    metrics: list[str] = Field(default_factory=list)
    action_items: list[str] = Field(default_factory=list)
    audience_questions: list[str] = Field(default_factory=list, description="Questions audience might ask")

analyzer = llm.with_structured_output(SlideAnalysis)

analyses = []
for slide in slides:
    analysis = analyzer.invoke(
        f"Analyze this presentation slide:\n"
        f"Title: {slide['title']}\nContent: {slide['content']}"
    )
    analyses.append(analysis)
    print(f"\nSlide {analysis.slide_number}: {analysis.title}")
    print(f"  Key points: {len(analysis.key_points)}")
    print(f"  Metrics: {analysis.metrics[:2]}")
    print(f"  Potential questions: {analysis.audience_questions[:2]}")

## Step 3 — Generate Speaker Script

In [ ]:
script_prompt = ChatPromptTemplate.from_messages([
    ("system", "Write a natural speaker script for presenting this slide. "
     "Include transitions, emphasis points, and timing notes. Target 60 seconds per slide."),
    ("human", "Slide {num}: {title}\nKey Points: {points}\n\nScript:")
])
script_chain = script_prompt | llm | StrOutputParser()

full_script = []
for analysis in analyses:
    script = script_chain.invoke({
        "num": analysis.slide_number,
        "title": analysis.title,
        "points": "; ".join(analysis.key_points[:4]),
    })
    full_script.append({"slide": analysis.slide_number, "script": script})
    print(f"\n--- Slide {analysis.slide_number} Script ---")
    print(script[:250] + "...")

## Step 4 — Q&A Preparation

In [ ]:
all_questions = []
for a in analyses:
    for q in a.audience_questions:
        all_questions.append({"slide": a.slide_number, "question": q})

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", "Prepare a concise, confident answer for a presentation Q&A."),
    ("human", "Context: {title}\nQuestion: {question}\nAnswer:")
])
qa_chain = qa_prompt | llm | StrOutputParser()

print("Q&A PREP SHEET")
print("=" * 50)
for qa in all_questions[:6]:
    slide = slides[qa["slide"] - 1]
    answer = qa_chain.invoke({"title": slide["title"], "question": qa["question"]})
    print(f"\n  Q: {qa['question']}")
    print(f"  A: {answer[:150]}")

## What You Learned
- **Slide deck analysis** with structured extraction
- **Speaker script generation** with timing
- **Q&A preparation** for anticipated questions
- **Presentation intelligence** from raw slide content